# 01 · Separability

Stage 1 analysis. Reads the gesture-probe dataset captured by the sensor harnesses (web / RN / iOS) into `gi_.session` + `gi_.sample`, and ultimately produces the separability report that drives the **Gate 4 decision** — auto-detect grip (Path A) vs. explicit instrument picker (Path B). See `docs/stage-1-protocol.md` for the 10-probe taxonomy and target counts (10 gestures × 10 reps × 3 platforms = 300 sessions).

**Connection.** Set `DATABASE_URL` in your shell to the Supabase Postgres connection string (Supabase dashboard → Project Settings → Database → Connection string → URI). The `gi_` schema is not exposed via PostgREST, so direct Postgres is the only read path.

**Deps (one-time):** `pip install pandas psycopg2-binary`

## Cell 1 — per-probe sample-count table

Answers "what data do we have right now?"

In [ ]:
import os
import pandas as pd
import psycopg2

DATABASE_URL = os.environ.get("DATABASE_URL")
if not DATABASE_URL:
    raise RuntimeError(
        "Set DATABASE_URL in your shell. Supabase dashboard → Settings → Database → "
        "Connection string (URI). Example: postgresql://postgres.<ref>:<pwd>"
        "@aws-0-ca-central-1.pooler.supabase.com:6543/postgres"
    )

QUERY = """
select
  gesture_label,
  platform,
  count(*)                              as n_sessions,
  sum(sample_count)                     as total_samples,
  round(avg(measured_hz)::numeric, 1)   as mean_hz,
  round(avg(duration_ms)::numeric, 0)   as mean_duration_ms,
  min(started_at)                       as first_seen,
  max(started_at)                       as last_seen
from gi_.session
group by gesture_label, platform
order by gesture_label, platform;
"""

with psycopg2.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute(QUERY)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]

df = pd.DataFrame.from_records(rows, columns=cols)
print(f"{len(df)} (gesture, platform) groups · {df['n_sessions'].sum()} sessions · {df['total_samples'].sum():,} samples total")
df

## Next cells (TODO)

- **Per-probe sensor stats:** mean / std / peak per axis of `accel`, `user_accel`, `gyro`; check noise-floor probes (`noise_floor_flat`, `noise_floor_held`) set the lower bound.
- **Dominant frequency:** FFT per session per axis; sanity-check that `shaker` lives at a higher band than `conducting_arc`.
- **Pairwise separability matrix:** cosine distance between mean feature vectors per gesture. Decision threshold for Gate 4 is **pairwise > 0.7** across grip-identifying probes (shaker / trombone / strum / arc / theremin variants).
- **Platform comparison:** does native's 100Hz reveal gestures that web's 60Hz buries? Produces the evidence on whether the web harness is good enough for Stage 2 production or only for prototyping.